In [117]:
import pandas as pd
import json
import numpy as np
import os

In [2]:
import re
def get_error_type(text):
    # Regex to match and extract 'AssertionError'
    match = re.search(r"type:\s*<class '([\w.]+)'>", text)
    
    if match:
        extracted_error = match.group(1)
        #if extracted_error == "AssertionError":
            #return "passed"
        #else:
        return extracted_error
    else:
        return text

In [3]:
import pickle
# Load the pipeline
with open('pipeline_model.pkl', 'rb') as f:
    pipeline = pickle.load(f)

/opt/homebrew/anaconda3/envs/interpret/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.4.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/homebrew/anaconda3/envs/interpret/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.4.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/homebrew/anaconda3/envs/interpret/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator Pipelin

In [80]:
df_filter_odex = read_filter()
df_filter_ds = read_filter('ds')

In [104]:
def filter_dataset(df, dataset='odex'):
    df_filter = read_filter(dataset)
    if dataset == 'odex':
        df_m = df[~df['id'].astype(str).isin(df_filter['id'].astype(str))]
    else:
        df_m = df[~df['completion_id'].isin(df_filter['completion_id'])]
    return df_m

In [13]:
def read_filter(dataset='odex'):
    if dataset == 'odex':
        df = pd.read_csv('results/results-camera/odex_ids_uncapable.csv')
    else:
        df = pd.read_csv('results/results-camera/ds_ids_uncapable.csv')
    return df
# Function to count passed
def count_passed(df):
    return len(df[df['execute_message'] == "passed"])
    
# Function to count assertion error
def count_assertion(df, bench='odex'):
    if bench == 'ds':
        return len(df[df['error_type'].str.contains("AssertionError")])
    elif bench == 'odex':
        return len(df[df['execute_message'] == "failed: "])
        
# Function to count original IFS
def count_ifs(df, dataset='odex'):
    # List of values to check
    values_list = ['TypeError', 'AttributeError', 'ModuleNotFoundError', 'ImportError']
    df_filter = read_filter(dataset)
    if dataset == 'odex':
        df_m = df[~df['id'].isin(df_filter['id'])]
    else:
        df_m = df[~df['completion_id'].isin(df_filter['completion_id'])]
    # Count rows where 'col1' contains values in values_list
    count = df_m['error_type'].isin(values_list).sum()
    return count

In [14]:
file_names_odex = ["results/odex/answers/claude-3-opus-20240229-0-answers.jsonl", 
              "results/odex/answers/gpt-3.5-turbo-0125-0-answers.jsonl",
              "results/odex/answers/gpt-4-0613-0-answers.jsonl",
              "results/odex/answers/granite-code-latest-0-answers.jsonl"]

In [15]:
file_names_ds = ["results/errors/claude-3-opus-20240229-0-errors.jsonl", 
                "results/errors/gpt-3.5-turbo-0125-0-errors.jsonl", 
              "results/errors/gpt-4-0613-0-errors.jsonl",
              "results/errors/granite-code-latest-no-grammar-0-errors.jsonl"]

In [16]:
for file in file_names_odex:
    print(file)
    file_df = pd.read_json(file,lines=True)
    file_df = file_df.rename(columns={'result': 'execute_message'})
    assertion_errors = count_assertion(file_df, 'odex')
    passed_rows = count_passed(file_df)
    file_df["error_type"] = pipeline.predict(file_df["execute_message"])
    print(f"ER: {assertion_errors+passed_rows}, passed: {passed_rows}, ifs: {count_ifs(file_df)}")

results/odex/answers/claude-3-opus-20240229-0-answers.jsonl
ER: 338, passed: 213, ifs: 73
results/odex/answers/gpt-3.5-turbo-0125-0-answers.jsonl
ER: 301, passed: 181, ifs: 52
results/odex/answers/gpt-4-0613-0-answers.jsonl
ER: 288, passed: 178, ifs: 44
results/odex/answers/granite-code-latest-0-answers.jsonl
ER: 234, passed: 91, ifs: 91


In [17]:
for file in file_names_ds:
    print(file)
    file_df = pd.read_json(file,lines=True)
    file_df = file_df.rename(columns={'result': 'execute_message'})
    file_df['error_type'] = file_df['execute_message'].apply(get_error_type)
    assertion_errors = count_assertion(file_df, 'ds')
    passed_rows = count_passed(file_df)
    print(f"ER: {assertion_errors+passed_rows}, passed: {passed_rows}, ifs: {count_ifs(file_df, dataset='ds')}")

results/errors/claude-3-opus-20240229-0-errors.jsonl
ER: 731, passed: 427, ifs: 80
results/errors/gpt-3.5-turbo-0125-0-errors.jsonl
ER: 732, passed: 366, ifs: 89
results/errors/gpt-4-0613-0-errors.jsonl
ER: 768, passed: 477, ifs: 69
results/errors/granite-code-latest-no-grammar-0-errors.jsonl
ER: 336, passed: 89, ifs: 98


In [ ]:
# Files to fix

In [95]:
# Enum completion_id
file_names_fix_odex_granite = ["results/classifier/annotated/granite-code-latest-general-0-answers.csv",
"results/classifier/annotated/granite-code-latest-union-0-answers.csv",
"results/classifier/annotated/granite-code-latest-mypy-0-answers.csv",
"results/classifier/annotated/granite-code-latest-pyright-0-answers.csv"]

file_names_fix_odex_o3 = ["results/classifier/annotated/o3-mini-general-0-answers.csv",
"results/classifier/annotated/o3-mini-union-0-answers.csv",
"results/classifier/annotated/o3-mini-mypy-0-answers.csv",
"results/classifier/annotated/o3-mini-pyright-0-answers.csv"]

In [96]:
ds1000 = pd.read_csv('DS1000_detector_al_errors.csv')
odex = pd.read_csv('Odex_detector_all_errors.csv')

In [97]:
ds1000['error_type'] = ds1000['execute_message'].apply(get_error_type)
ds1000_error = ds1000[ds1000['execute']==0]
ds1000_error['enum_id'] = ds1000_error.groupby('completion_id').cumcount()+ 1

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_72071/3349914892.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds1000_error['enum_id'] = ds1000_error.groupby('completion_id').cumcount()+ 1


In [98]:
odex["error_type"] = pipeline.predict(odex["execute_message"])
odex_error = odex[odex['execute']==0]
odex_error[['model', 'id']] = odex_error['completion_id'].str.extract(r"(.+)[-_](?:0_)?(\d+)\.py")
# Enumerate occurrences of the same 'after' value
odex_error['enum_id'] = odex_error.groupby('id').cumcount()+ 1

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_72071/2916036304.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  odex_error[['model', 'id']] = odex_error['completion_id'].str.extract(r"(.+)[-_](?:0_)?(\d+)\.py")
/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_72071/2916036304.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  odex_error[['model', 'id']] = odex_error['completion_id'].str.extract(r"(.+)[-_](?:0_)?(\d+)\.py")
/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel

In [99]:
odex_error['mypy'] = odex_error['mypy_str'].apply(lambda x : 1 if isinstance(x, str) else 0)
odex_error['pyright'] = odex_error['pyright_str'].apply(lambda x : 1 if isinstance(x, str) else 0)

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_72071/3376517216.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  odex_error['mypy'] = odex_error['mypy_str'].apply(lambda x : 1 if isinstance(x, str) else 0)
/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_72071/3376517216.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  odex_error['pyright'] = odex_error['pyright_str'].apply(lambda x : 1 if isinstance(x, str) else 0)


In [107]:
505-len(df_filter_odex)

466

In [102]:
subset_odex = odex_error[odex_error['error_type'].isin(filter_values)]
subset_odex.groupby(['model'])['completion_id'].count()*(100/505)

model
claude-3-opus-20240229-0    20.396040
gpt-3.5-0125                15.841584
gpt-4-0613                  13.663366
granite-code-latest-0       23.960396
Name: completion_id, dtype: float64

In [108]:
subset_odex_amb = filter_dataset(subset_odex, 'odex')
subset_odex_amb.groupby(['model'])['completion_id'].count()*(100/466)

model
claude-3-opus-20240229-0    15.665236
gpt-3.5-0125                11.158798
gpt-4-0613                   9.442060
granite-code-latest-0       19.527897
Name: completion_id, dtype: float64

In [111]:
df_o3_odex = pd.DataFrame()
for file in file_names_fix_odex_o3:
    odex_csv = pd.read_csv(file)
    odex_csv['enum_id'] = odex_csv.groupby('id').cumcount()+ 1
    odex_csv['id'] = odex_csv['id'].astype(str)
    merge_df = pd.merge(odex_csv[['id','prediction','enum_id']],odex_error[['model','id','enum_id']], on=['id','enum_id'], how='left')
    merge_df = filter_dataset(merge_df, 'odex')
    # List of prediction values to keep
    #filter_values = ['AssertionError', 'AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError', 'passed']  # Only keep rows where prediction is 1 or 3
    filter_values = ['AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError']  # Only keep rows where prediction is 1 or 3
    # Filter before grouping
    filtered_df = merge_df[merge_df['prediction'].isin(filter_values)]
    not_in_filter = merge_df[~merge_df['prediction'].isin(['AssertionError', 'passed'])]
    #filtered_df = filter_dataset(filtered_df, 'odex')
    #filtered_df.groupby(['model','prediction'])['id'].count()
    gdf = filtered_df.groupby(['model','prediction'])['id'].count().reset_index()
    gdf['label'] = file
    gdf['bench'] = 'odex'
    df_o3_odex = pd.concat([df_o3_odex, gdf])
#df_o3_odex.to_csv('o3_mini_general_odex.csv', index=False)

In [112]:
df_o3_odex.groupby(['model','label'])['id'].sum()

model                     label                                                     
claude-3-opus-20240229-0  results/classifier/annotated/o3-mini-general-0-answers.csv    47
                          results/classifier/annotated/o3-mini-mypy-0-answers.csv       44
                          results/classifier/annotated/o3-mini-pyright-0-answers.csv    46
                          results/classifier/annotated/o3-mini-union-0-answers.csv      46
gpt-3.5-0125              results/classifier/annotated/o3-mini-general-0-answers.csv    49
                          results/classifier/annotated/o3-mini-mypy-0-answers.csv       45
                          results/classifier/annotated/o3-mini-pyright-0-answers.csv    49
                          results/classifier/annotated/o3-mini-union-0-answers.csv      40
gpt-4-0613                results/classifier/annotated/o3-mini-general-0-answers.csv    27
                          results/classifier/annotated/o3-mini-mypy-0-answers.csv       22
     

In [110]:
df_o3_odex.groupby(['model','label'])['id'].sum()

model                     label                                                     
claude-3-opus-20240229-0  results/classifier/annotated/o3-mini-general-0-answers.csv    67
                          results/classifier/annotated/o3-mini-mypy-0-answers.csv       62
                          results/classifier/annotated/o3-mini-pyright-0-answers.csv    65
                          results/classifier/annotated/o3-mini-union-0-answers.csv      62
gpt-3.5-0125              results/classifier/annotated/o3-mini-general-0-answers.csv    64
                          results/classifier/annotated/o3-mini-mypy-0-answers.csv       58
                          results/classifier/annotated/o3-mini-pyright-0-answers.csv    66
                          results/classifier/annotated/o3-mini-union-0-answers.csv      53
gpt-4-0613                results/classifier/annotated/o3-mini-general-0-answers.csv    33
                          results/classifier/annotated/o3-mini-mypy-0-answers.csv       25
     

In [118]:
# Map label to short name
label_map = {
    'results/classifier/annotated/o3-mini-general-0-answers.csv': 'general',
    'results/classifier/annotated/o3-mini-mypy-0-answers.csv':    'mypy',
    'results/classifier/annotated/o3-mini-pyright-0-answers.csv': 'pyright',
    'results/classifier/annotated/o3-mini-union-0-answers.csv':   'union',
}

df_o3_odex['label_short'] = df_o3_odex['label'].map(label_map)

# Pivot so each label becomes a column
o3_result = df_o3_odex.groupby(['model', 'label_short'])['id'].sum().unstack('label_short').reset_index()

# Normalize against the general counts from earlier (result df)
for col in ['mypy', 'pyright', 'general']:
    o3_result[col] = (o3_result[col] / 466) * 100
    #o3_result[col] = (o3_result[col] / summary_ds_errors.set_index('model')[col].values) * 100

In [119]:
o3_result

label_short,model,general,mypy,pyright,union
0,claude-3-opus-20240229-0,10.085837,9.442060,9.871245,46
1,gpt-3.5-0125,10.515021,9.656652,10.515021,40
2,gpt-4-0613,5.793991,4.721030,5.793991,19
3,granite-code-latest-0,2.789700,1.502146,3.004292,8


In [140]:
df_o3_odex = pd.DataFrame()
for file in file_names_fix_odex_o3:
    odex_csv = pd.read_csv(file)
    odex_csv['enum_id'] = odex_csv.groupby('id').cumcount() + 1
    odex_csv['id'] = odex_csv['id'].astype(str)
    merge_df = pd.merge(odex_csv[['id','prediction','enum_id']], odex_error[['model','id','enum_id']], on=['id','enum_id'], how='left')
    #merge_df = filter_dataset(merge_df, 'odex')

    filter_values = ['AttributeError', 'TypeError', 'ModuleNotFoundError', 'ImportError']
    filtered_df   = merge_df[merge_df['prediction'].isin(filter_values)]
    filtered_df = filter_dataset(filtered_df, 'odex')
    not_in_filter = merge_df[~merge_df['prediction'].isin(['AssertionError', 'passed'])]

    # Count per model
    filtered_count      = filtered_df.groupby('model')['id'].count().rename('filtered')
    not_in_filter_count = not_in_filter.groupby('model')['id'].count().rename('not_in_filter')

    gdf = pd.concat([filtered_count, not_in_filter_count], axis=1).reset_index()
    gdf['ratio_norm']         = (gdf['filtered'] / gdf['not_in_filter']) * 100   # % of non-passing that are filtered errors
    gdf['norm_filtered']      = (gdf['filtered'] / 466) * 100                    # % of total examples that are filtered errors
    gdf['norm_not_in_filter'] = (gdf['not_in_filter'] / 466) * 100               # % of total examples that are non-passing
    gdf['label'] = file
    gdf['bench'] = 'odex'

    df_o3_odex = pd.concat([df_o3_odex, gdf])

In [141]:
label_map = {
    'results/classifier/annotated/o3-mini-general-0-answers.csv': 'general',
    'results/classifier/annotated/o3-mini-mypy-0-answers.csv':    'mypy',
    'results/classifier/annotated/o3-mini-pyright-0-answers.csv': 'pyright',
    'results/classifier/annotated/o3-mini-union-0-answers.csv':   'union',
}

df_o3_odex['label_short'] = df_o3_odex['label'].map(label_map)

metrics = ['ratio_norm', 'norm_filtered', 'norm_not_in_filter']

pivots = []
for metric in metrics:
    pivot = df_o3_odex.groupby(['model', 'label_short'])[metric].mean().unstack('label_short')
    pivot.columns = [f'{c}_{metric}' for c in pivot.columns]
    pivots.append(pivot)

o3_result = pd.concat(pivots, axis=1).reset_index()

In [142]:
o3_result

,model,general_ratio_norm,mypy_ratio_norm,pyright_ratio_norm,union_ratio_norm,general_norm_filtered,mypy_norm_filtered,pyright_norm_filtered,union_norm_filtered,general_norm_not_in_filter,mypy_norm_not_in_filter,pyright_norm_not_in_filter,union_norm_not_in_filter
0,claude-3-opus-20240229-0,43.925234,44.000000,45.098039,48.421053,10.085837,9.442060,9.871245,9.871245,22.961373,21.459227,21.888412,20.386266
1,gpt-3.5-0125,44.144144,47.368421,45.794393,43.010753,10.515021,9.656652,10.515021,8.583691,23.819742,20.386266,22.961373,19.957082
2,gpt-4-0613,54.000000,70.967742,58.695652,67.857143,5.793991,4.721030,5.793991,4.077253,10.729614,6.652361,9.871245,6.008584
3,granite-code-latest-0,54.166667,70.000000,58.333333,88.888889,2.789700,1.502146,3.004292,1.716738,5.150215,2.145923,5.150215,1.931330


In [23]:
df_o3_odex = pd.DataFrame()
for file in file_names_fix_odex_granite:
    odex_csv = pd.read_csv(file)
    odex_csv['enum_id'] = odex_csv.groupby('id').cumcount()+ 1
    odex_csv['id'] = odex_csv['id'].astype(str)
    merge_df = pd.merge(odex_csv[['id','prediction','enum_id']],odex_error[['model','id','enum_id','error_type']], on=['id','enum_id'], how='left')
    # List of prediction values to keep
    #filter_values = ['AssertionError', 'AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError', 'passed']  # Only keep rows where prediction is 1 or 3
    filter_values = ['AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError']  # Only keep rows where prediction is 1 or 3
    # Filter before grouping
    filtered_df_repair = merge_df[merge_df['prediction'].isin(filter_values)]
    gdf = filtered_df_repair.groupby(['model','prediction'])['id'].count().reset_index()
    gdf = gdf.groupby(['model'])['id'].sum().reset_index()
    # Filter total
    filtered_df_total = merge_df[merge_df['error_type'].isin(filter_values)]
    gdf_total = filtered_df_total.groupby(['model','error_type'])['id'].count().reset_index()
    gdf_total = gdf_total.groupby(['model'])['id'].sum().reset_index()
    merge_df_new = pd.merge(gdf_total, gdf, on=['model'], how='left', suffixes=("_before","_after"))
    merge_df_new['label'] = file
    merge_df_new['bench'] = 'odex'
    df_o3_odex = pd.concat([df_o3_odex, merge_df_new])
#df_o3_odex.to_csv('granite_general_odex_ifs.csv', index=False)

In [106]:
df_o3_odex = pd.DataFrame()
for file in file_names_fix_odex_granite:
    odex_csv = pd.read_csv(file)
    odex_csv['enum_id'] = odex_csv.groupby('id').cumcount()+ 1
    odex_csv['id'] = odex_csv['id'].astype(str)
    merge_df = pd.merge(odex_csv[['id','prediction','enum_id']],odex_error[['model','id','enum_id']], on=['id','enum_id'], how='left')
    # List of prediction values to keep
    filter_values = ['AssertionError', 'AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError', 'passed']  # Only keep rows where prediction is 1 or 3
    # Filter before grouping
    filtered_df = merge_df[merge_df['prediction'].isin(filter_values)]
    #filtered_df.groupby(['model','prediction'])['id'].count()
    gdf = filtered_df.groupby(['model','prediction'])['id'].count().reset_index()
    gdf['label'] = file
    gdf['bench'] = 'granite'
    df_o3_odex = pd.concat([df_o3_odex, gdf])
#df_o3_odex.to_csv('o3_mini_general_granite.csv', index=False)

In [126]:
# Enum id
file_names_fix_ds_o3 = ['results/errors/o3/o3-mini-repair-pyright-0-errors.jsonl',
'results/errors/o3/o3-mini-repair-union-0-errors.jsonl',
'results/errors/o3/o3-mini-repair-mypy-0-errors.jsonl',
'results/errors/o3/o3-mini-repair-general-0-errors.jsonl',]

file_names_fix_ds_granite = ['results/errors/ibm/repair/granite-code-latest-repair-pyright-0-errors.jsonl',
'results/errors/ibm/repair/granite-code-latest-repair-union-0-errors.jsonl',
'results/errors/ibm/repair/granite-code-latest-repair-mypy-0-errors.jsonl',
'results/errors/ibm/repair/granite-code-latest-repair-general-0-errors.jsonl',]

In [127]:
odex_csv = pd.read_json('results/errors/o3/o3-mini-repair-pyright-0-errors.jsonl', lines=True)
odex_csv['enum_id'] = odex_csv.groupby('completion_id').cumcount()+ 1
odex_csv

,passed,result,completion_id,generate_code,score,library,perturbation_type,enum_id
0,False,failed: expected an indented block after funct...,12,import pandas as pd\nimport numpy as np\nimpor...,0,Pandas,Surface,1
1,False,"failed: 'result', type: <class 'KeyError'>",0,import pandas as pd\nimport numpy as np\nimpor...,0,Pandas,Origin,1
2,False,"failed: , type: <class 'AssertionError'>",28,import pandas as pd\nimport numpy as np\nimpor...,0,Pandas,Difficult-Rewrite,1
3,False,failed: expected an indented block after funct...,4,import pandas as pd\nimport numpy as np\nimpor...,0,Pandas,Surface,1
4,False,failed: expected an indented block after funct...,4,import pandas as pd\nimport numpy as np\nimpor...,0,Pandas,Surface,2
...,...,...,...,...,...,...,...,...
821,False,failed: name 'confidence_score' is not defined...,993,import torch\nimport copy\nimport sklearn\nfro...,0,Pytorch,Origin,3
822,False,failed: Sizes of tensors must match except in ...,998,import torch\nimport copy\n\n\ndef generate_te...,0,Pytorch,Semantic,1
823,False,"failed: , type: <class 'AssertionError'>",993,import torch\nimport copy\nimport sklearn\nfro...,0,Pytorch,Origin,4
824,False,failed: Sizes of tensors must match except in ...,998,import torch\nimport copy\n\n\ndef generate_te...,0,Pytorch,Semantic,2


In [128]:
odex_errors = pd.read_json('results/results-camera/o3-mini/repair/odex_repair.jsonl', lines=True)
ds_errors = pd.read_json('results/results-camera/o3-mini/repair/ds_repair.jsonl', lines=True)

In [129]:
ds1000_error['mypy'] = ds1000_error['mypy_str'].apply(lambda x : 1 if isinstance(x, str) else 0)
ds1000_error['pyright'] = ds1000_error['pyright_str'].apply(lambda x : 1 if isinstance(x, str) else 0)

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_72071/3814523845.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds1000_error['mypy'] = ds1000_error['mypy_str'].apply(lambda x : 1 if isinstance(x, str) else 0)
/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_72071/3814523845.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds1000_error['pyright'] = ds1000_error['pyright_str'].apply(lambda x : 1 if isinstance(x, str) else 0)


In [61]:
# Get general count
general = ds1000_error.groupby('model')['completion_id'].count().rename('general')

# Get mypy=1 count
mypy = ds1000_error[ds1000_error['mypy'] == 1].groupby('model')['completion_id'].count().rename('mypy')

# Get pyright=1 count
pyright = ds1000_error[ds1000_error['pyright'] == 1].groupby('model')['completion_id'].count().rename('pyright')

# Combine
summary_ds_errors = pd.concat([general, mypy, pyright], axis=1).reset_index()

In [84]:
subset_ds1000 = ds1000_error[ds1000_error['error_type'].isin(filter_values)]
subset_ds1000_amb = filter_dataset(subset_ds1000, 'ds')

In [106]:
subset_ds1000_amb.groupby(['model'])['completion_id'].count()*(100/970)

model
claude-3-opus-20240229-0             8.247423
gpt-3.5-0125-0                       9.175258
gpt-4-0613                           7.113402
granite-code-latest-no-grammar-0    10.103093
Name: completion_id, dtype: float64

In [78]:
subset_ds1000.groupby(['model'])['completion_id'].count()/10

model
claude-3-opus-20240229-0             9.1
gpt-3.5-0125-0                       9.7
gpt-4-0613                           8.1
granite-code-latest-no-grammar-0    11.4
Name: completion_id, dtype: float64

In [86]:
df_o3_ds = pd.DataFrame()
for file in file_names_fix_ds_o3:
    odex_csv = pd.read_json(file, lines=True)
    odex_csv['enum_id'] = odex_csv.groupby('completion_id').cumcount()+ 1
    odex_csv['error_type'] = odex_csv['result'].apply(get_error_type)
    merge_df = pd.merge(odex_csv[['completion_id','error_type','enum_id']],ds1000_error[['model','completion_id','enum_id']], on=['completion_id','enum_id'], how='left')
    # List of prediction values to keep
    filter_values = ['AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError']  # Only keep rows where prediction is 1 or 3
    # Filter before grouping
    filtered_df = merge_df[merge_df['error_type'].isin(filter_values)]
    filtered_df = filter_dataset(filtered_df, 'ds')
    #filtered_df.groupby(['model','prediction'])['id'].count()
    gdf = filtered_df.groupby(['model','error_type'])['completion_id'].count().reset_index()
    gdf['label'] = file
    gdf['bench'] = 'ds'
    gdf['model_bench'] = 'o3'
    df_o3_ds = pd.concat([df_o3_ds, gdf])
#df_o3_ds.to_csv('o3_mini_general_ds.csv', index=False)

In [87]:
df_o3_ds.groupby(['model','label'])['completion_id'].sum()

model                             label                                                  
claude-3-opus-20240229-0          results/errors/o3/o3-mini-repair-general-0-errors.jsonl    18
                                  results/errors/o3/o3-mini-repair-mypy-0-errors.jsonl       12
                                  results/errors/o3/o3-mini-repair-pyright-0-errors.jsonl    17
                                  results/errors/o3/o3-mini-repair-union-0-errors.jsonl      14
gpt-3.5-0125-0                    results/errors/o3/o3-mini-repair-general-0-errors.jsonl    21
                                  results/errors/o3/o3-mini-repair-mypy-0-errors.jsonl       13
                                  results/errors/o3/o3-mini-repair-pyright-0-errors.jsonl    14
                                  results/errors/o3/o3-mini-repair-union-0-errors.jsonl      15
gpt-4-0613                        results/errors/o3/o3-mini-repair-general-0-errors.jsonl    15
                                  results/erro

In [64]:
df_o3_ds.groupby(['model','label'])['completion_id'].sum()

model                             label                                                  
claude-3-opus-20240229-0          results/errors/o3/o3-mini-repair-general-0-errors.jsonl    27
                                  results/errors/o3/o3-mini-repair-mypy-0-errors.jsonl       17
                                  results/errors/o3/o3-mini-repair-pyright-0-errors.jsonl    22
                                  results/errors/o3/o3-mini-repair-union-0-errors.jsonl      19
gpt-3.5-0125-0                    results/errors/o3/o3-mini-repair-general-0-errors.jsonl    26
                                  results/errors/o3/o3-mini-repair-mypy-0-errors.jsonl       19
                                  results/errors/o3/o3-mini-repair-pyright-0-errors.jsonl    18
                                  results/errors/o3/o3-mini-repair-union-0-errors.jsonl      19
gpt-4-0613                        results/errors/o3/o3-mini-repair-general-0-errors.jsonl    19
                                  results/erro

In [91]:
1000-len(df_filter_ds)

970

In [92]:
# Map label to short name
label_map = {
    'results/errors/o3/o3-mini-repair-general-0-errors.jsonl': 'general',
    'results/errors/o3/o3-mini-repair-mypy-0-errors.jsonl':    'mypy',
    'results/errors/o3/o3-mini-repair-pyright-0-errors.jsonl': 'pyright',
    'results/errors/o3/o3-mini-repair-union-0-errors.jsonl':   'union',
}

df_o3_ds['label_short'] = df_o3_ds['label'].map(label_map)

# Pivot so each label becomes a column
o3_result = df_o3_ds.groupby(['model', 'label_short'])['completion_id'].sum().unstack('label_short').reset_index()

# Normalize against the general counts from earlier (result df)
for col in ['mypy', 'pyright', 'general']:
    o3_result[col] = (o3_result[col] / 970) * 100
    #o3_result[col] = (o3_result[col] / summary_ds_errors.set_index('model')[col].values) * 100

In [93]:
o3_result

label_short,model,general,mypy,pyright,union
0,claude-3-opus-20240229-0,1.855670,1.237113,1.752577,14
1,gpt-3.5-0125-0,2.164948,1.340206,1.443299,15
2,gpt-4-0613,1.546392,0.618557,0.927835,10
3,granite-code-latest-no-grammar-0,4.123711,1.855670,1.958763,23


In [137]:
df_o3_ds = pd.DataFrame()
for file in file_names_fix_ds_o3:
    odex_csv = pd.read_json(file, lines=True)
    odex_csv['enum_id'] = odex_csv.groupby('completion_id').cumcount()+ 1
    odex_csv['error_type'] = odex_csv['result'].apply(get_error_type)
    merge_df = pd.merge(odex_csv[['completion_id','error_type','enum_id']],ds1000_error[['model','completion_id','enum_id']], on=['completion_id','enum_id'], how='left')
    # List of prediction values to keep
    
    filter_values = ['AttributeError', 'TypeError', 'ModuleNotFoundError', 'ImportError']
    filtered_df   = merge_df[merge_df['error_type'].isin(filter_values)]
    filtered_df = filter_dataset(filtered_df, 'ds')
    not_in_filter = merge_df[~merge_df['error_type'].isin(['AssertionError', 'passed'])]

    # Count per model
    filtered_count      = filtered_df.groupby('model')['completion_id'].count().rename('filtered')
    not_in_filter_count = not_in_filter.groupby('model')['completion_id'].count().rename('not_in_filter')

    gdf = pd.concat([filtered_count, not_in_filter_count], axis=1).reset_index()
    gdf['ratio_norm']         = (gdf['filtered'] / gdf['not_in_filter']) * 100   # % of non-passing that are filtered errors
    gdf['norm_filtered']      = (gdf['filtered'] / 970) * 100                    # % of total examples that are filtered errors
    gdf['norm_not_in_filter'] = (gdf['not_in_filter'] / 970) * 100               # % of total examples that are non-passing
    gdf['label'] = file
    gdf['bench'] = 'odex'

    df_o3_ds = pd.concat([df_o3_ds, gdf])

In [138]:
# Map label to short name
label_map = {
    'results/errors/o3/o3-mini-repair-general-0-errors.jsonl': 'general',
    'results/errors/o3/o3-mini-repair-mypy-0-errors.jsonl':    'mypy',
    'results/errors/o3/o3-mini-repair-pyright-0-errors.jsonl': 'pyright',
    'results/errors/o3/o3-mini-repair-union-0-errors.jsonl':   'union',
}

df_o3_ds['label_short'] = df_o3_ds['label'].map(label_map)

metrics = ['ratio_norm', 'norm_filtered', 'norm_not_in_filter']

pivots = []
for metric in metrics:
    pivot = df_o3_ds.groupby(['model', 'label_short'])[metric].mean().unstack('label_short')
    pivot.columns = [f'{c}_{metric}' for c in pivot.columns]
    pivots.append(pivot)

o3_result = pd.concat(pivots, axis=1).reset_index()


In [139]:
o3_result

,model,general_ratio_norm,mypy_ratio_norm,pyright_ratio_norm,union_ratio_norm,general_norm_filtered,mypy_norm_filtered,pyright_norm_filtered,union_norm_filtered,general_norm_not_in_filter,mypy_norm_not_in_filter,pyright_norm_not_in_filter,union_norm_not_in_filter
0,claude-3-opus-20240229-0,11.042945,9.677419,13.600000,11.475410,1.855670,1.237113,1.752577,1.443299,16.804124,12.783505,12.886598,12.577320
1,gpt-3.5-0125-0,13.125000,11.111111,12.844037,13.392857,2.164948,1.340206,1.443299,1.546392,16.494845,12.061856,11.237113,11.546392
2,gpt-4-0613,11.194030,6.451613,10.112360,11.111111,1.546392,0.618557,0.927835,1.030928,13.814433,9.587629,9.175258,9.278351
3,granite-code-latest-no-grammar-0,9.389671,7.468880,9.405941,11.616162,4.123711,1.855670,1.958763,2.371134,43.917526,24.845361,20.824742,20.412371


In [120]:
df_o3_granite = pd.DataFrame()
for file in file_names_fix_ds_granite :
    odex_csv = pd.read_json(file, lines=True)
    odex_csv['enum_id'] = odex_csv.groupby('completion_id').cumcount()+ 1
    odex_csv['error_type'] = odex_csv['result'].apply(get_error_type)
    merge_df = pd.merge(odex_csv[['completion_id','error_type','enum_id']],ds1000_error[['model','completion_id','enum_id']], on=['completion_id','enum_id'], how='left')
    # List of prediction values to keep
    filter_values = ['AssertionError', 'AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError', 'passed']  # Only keep rows where prediction is 1 or 3
    # Filter before grouping
    filtered_df = merge_df[merge_df['error_type'].isin(filter_values)]
    #filtered_df.groupby(['model','prediction'])['id'].count()
    gdf = filtered_df.groupby(['model','error_type'])['completion_id'].count().reset_index()
    gdf['label'] = file
    gdf['bench'] = 'ds'
    gdf['model_bench'] = 'o3'
    df_o3_granite = pd.concat([df_o3_granite, gdf])
#df_o3_granite.to_csv('granite_mini_general_ds.csv', index=False)

In [146]:
df_o3_ds = pd.DataFrame()
for file in file_names_fix_ds_granite:
    odex_csv = pd.read_json(file, lines=True)
    odex_csv['enum_id'] = odex_csv.groupby('completion_id').cumcount()+ 1
    odex_csv['error_type'] = odex_csv['result'].apply(get_error_type)
    merge_df = pd.merge(odex_csv[['completion_id','error_type','enum_id']],ds1000_error[['model','completion_id','enum_id','error_type']], on=['completion_id','enum_id'], how='left')
    # List of prediction values to keep
    filter_values = ['AttributeError', 'TypeError','ModuleNotFoundError', 'ImportError']  # Only keep rows where prediction is 1 or 3
    # Filter before grouping
    filtered_df_repair = merge_df[merge_df['error_type_x'].isin(filter_values)]
    gdf = filtered_df_repair.groupby(['model','error_type_x'])['completion_id'].count().reset_index()
    gdf = gdf.groupby(['model'])['completion_id'].sum().reset_index()
    # Filter total
    filtered_df_total = merge_df[merge_df['error_type_y'].isin(filter_values)]
    gdf_total = filtered_df_total.groupby(['model','error_type_y'])['completion_id'].count().reset_index()
    gdf_total = gdf_total.groupby(['model'])['completion_id'].sum().reset_index()
    merge_df_new = pd.merge(gdf_total, gdf, on=['model'], how='left', suffixes=("_before","_after"))
    merge_df_new = pd.merge(gdf_total, gdf, on=['model'], how='left', suffixes=("_before","_after"))
    merge_df_new['label'] = file
    merge_df_new['bench'] = 'odex'
    df_o3_ds = pd.concat([df_o3_ds, merge_df_new])
#df_o3_ds.to_csv('granite_general_ds_ifs.csv', index=False)

In [65]:
odex_csv = pd.read_csv('results/classifier/annotated/o3-mini-general-0-answers.csv')
odex_csv['enum_id'] = odex_csv.groupby('id').cumcount()+ 1

In [68]:
pd.merge(odex_csv[['id','prediction','enum_id']],odex_error[['model','id','enum_id','error_type']], on=['id','enum_id'], how='left').groupby(['model','prediction']).count()

id  enum_id  error_type
model                    prediction                                  
claude-3-opus-20240229-0 AssertionError       10       10          10
                         AttributeError       23       23          23
                         FileNotFoundError     2        2           2
                         IndexError            2        2           2
                         KeyError             12       12          12
                         ModuleNotFoundError   3        3           3
                         NameError             4        4           4
                         TestCaseError         1        1           1
                         TypeError            41       41          41
                         ValueError           14       14          14
                         passed                7        7           7
                         timed out             4        4           4
gpt-3.5-0125             AssertionError       16       16          16
                         AttributeError       22       22          22
                         FileNotFoundError     3        3           3
                         IndexError            5        5           5
                         KeyError              9        9           9
                         ModuleNotFoundError   5        5           5
                         NameError             3        3           3
                         SyntaxError           1        1           1
                         TestCaseError         1        1           1
                         TypeError            37       37          37
                         ValueError           16       16          16
                         passed               23       23          23
                         timed out             9        9           9
gpt-4-0613               AssertionError       13       13          13
                         AttributeError       12       12          12
                         IndexError            1        1           1
                         KeyError              6        6           6
                         ModuleNotFoundError   1        1           1
                         NameError             1        1           1
                         TypeError            20       20          20
                         ValueError            4        4           4
                         passed               40       40          40
                         timed out             5        5           5
granite-code-latest-0    AssertionError       12       12          12
                         AttributeError        2        2           2
                         ImportError           1        1           1
                         KeyError              2        2           2
                         ModuleNotFoundError   2        2           2
                         NameError             1        1           1
                         SyntaxError           1        1           1
                         TypeError             9        9           9
                         ValueError            3        3           3
                         passed                8        8           8
                         timed out             3        3           3